# 03. Stage 1B: Collaborative Filtering (Implicit ALS)

Notebook này xây dựng tầng lọc thô dựa trên hành vi tương tác cộng tác của người dùng (Collaborative Filtering) thông qua dữ liệu ngầm định (Implicit Feedback).

---

### Phân tích Quyết định Thiết kế:
*   **Tại sao chọn Implicit ALS (iALS)?**
    *   Trong thực tế, dữ liệu tương tác của người dùng chủ yếu là ngầm định (click, xem phim, tìm kiếm) chứ không có ratings tường minh. Nếu ta áp dụng SVD truyền thống, ta buộc phải coi phim chưa xem là nhãn âm (0), điều này sai vì có thể họ chưa biết phim đó. **Implicit ALS** giải quyết triệt để bằng cách xem tất cả tương tác là thước đo độ tin cậy (confidence matrix) kết hợp với các latent factors để mô hình hóa sở thích ẩn.
*   **Tại sao không chọn BPR-MF làm giải thuật chính ở Retrieval?**
    *   BPR-MF tối ưu hóa ranking cặp (pairwise) rất tốt cho danh sách ngắn, nhưng iALS hoạt động dựa trên toàn bộ ma trận (pointwise matrix factorization với confidence weights), giúp tận dụng tối đa tần suất và loại sự kiện khác nhau (click vs watch_complete) dễ dàng hơn. BPR chỉ nhận nhãn nhị phân (1/0).
*   **Tại sao không chọn Collaborative Filtering dựa trên lân cận (KNN)?**
    *   KNN yêu cầu lưu trữ và tính toán ma trận tương tương giữa tất cả các cặp User hoặc Item ($O(N^2)$ hoặc $O(M^2)$). Điều này gây tốn bộ nhớ nghiêm trọng và không thể mở rộng (scale) khi hệ thống đạt hàng chục nghìn người dùng.


### Bước 1: Khởi tạo và Mã hóa Index
Tế bào này tải dữ liệu hành vi click giả lập và phim cào được. Vì các thuật toán Matrix Factorization trong thư viện `implicit` yêu cầu ma trận thưa với chỉ mục liên tục, ta thực hiện lập bản đồ (encode) các giá trị ID gốc (`userId`, `movieId`) sang chỉ mục nguyên liên tục bắt đầu từ 0. Dict mapping này được lưu vào file `id_mappings.pkl`.


In [1]:
import os
import pandas as pd
import numpy as np
import scipy.sparse as sp
import implicit
import pickle

# Load dữ liệu tương tác ngầm định đã lọc sạch rò rỉ (leakage) và ratings
clicks_df = pd.read_csv("processed_data/train_clicks.csv")
train_ratings = pd.read_csv("processed_data/train_ratings.csv")
movies_df = pd.read_csv(os.path.join("..", "..", "data", "crawler", "movies_crawled.csv"))

# Encode userId và movieId sang dạng index liên tục
unique_users = clicks_df['userId'].unique()
unique_movies = movies_df['movieId'].unique()

user_to_idx = {uid: i for i, uid in enumerate(unique_users)}
movie_to_idx = {mid: i for i, mid in enumerate(unique_movies)}
idx_to_movie = {i: mid for mid, i in movie_to_idx.items()}

# Lưu dict map để dùng lại
os.makedirs("processed_data", exist_ok=True)
with open("processed_data/id_mappings.pkl", "wb") as f:
    pickle.dump((user_to_idx, movie_to_idx, idx_to_movie), f)


### Bước 2: Xây dựng Ma trận Tương tác Thưa có Trọng số
Hiệu quả của iALS phụ thuộc vào việc xây dựng ma trận trọng số (confidence score).
Chúng ta thiết lập trọng số phễu hành vi (Behavior Funnel Weighting) cho cả click events và rating events:
*   `click`: **1.0**
*   `detail_view`: **2.0**
*   `watch_start`: **3.0**
*   `watch_complete` hoặc `Explicit rating >= 3.5`: **5.0** (Tín hiệu xem hết/rất thích)
*   `Explicit rating < 3.5`: **1.5** (Tín hiệu tương tác nhưng không thích lắm)

Ta kết hợp và lấy trọng số lớn nhất (`max()`) cho mỗi cặp User-Movie để làm đầu vào ma trận thưa.


In [2]:
# 1. Xây dựng ma trận tương tác có trọng số từ Behavior Funnel và Ratings
event_weights = {
    'click': 1.0,
    'detail_view': 2.0,
    'watch_start': 3.0,
    'watch_complete': 5.0
}

clicks_df['weight'] = clicks_df['event_type'].map(event_weights)
user_movie_weights = clicks_df.groupby(['userId', 'movieId'])['weight'].sum().reset_index()

# Tích hợp rating (explicit feedback) vào ma trận
ratings_weights = train_ratings.copy()
ratings_weights['weight'] = ratings_weights['rating'].apply(lambda r: 5.0 if r >= 3.5 else 1.5)
ratings_weights = ratings_weights[['userId', 'movieId', 'weight']]

# Gộp cả 2 nguồn, lấy giá trị lớn nhất cho mỗi cặp User-Movie
combined_weights = pd.concat([user_movie_weights, ratings_weights], ignore_index=True)
user_movie_weights = combined_weights.groupby(['userId', 'movieId'])['weight'].max().reset_index()

user_movie_weights = user_movie_weights[user_movie_weights['movieId'].isin(movie_to_idx.keys())]
user_movie_weights = user_movie_weights[user_movie_weights['userId'].isin(user_to_idx.keys())]

user_indices = user_movie_weights['userId'].map(user_to_idx).values
item_indices = user_movie_weights['movieId'].map(movie_to_idx).values
weights = user_movie_weights['weight'].values

num_users = len(user_to_idx)
num_items = len(movie_to_idx)

user_item_matrix = sp.csr_matrix((weights, (user_indices, item_indices)), shape=(num_users, num_items))
print(f"Sparse matrix density: {100 * user_item_matrix.nnz / (num_users * num_items):.4f}%")


Sparse matrix density: 0.0658%


### Bước 3: Huấn luyện mô hình Implicit ALS (Alternating Least Squares)

#### Nguyên lý Toán học của mô hình iALS:
Mô hình iALS phân rã ma trận tương tác người dùng - vật phẩm $R$ thành hai ma trận nhân tử ẩn có số chiều thấp: Ma trận User Factors $X \in \mathbb{R}^{U \times f}$ và Ma trận Item Factors $Y \in \mathbb{R}^{I \times f}$. 
Hàm mục tiêu tối ưu hóa bình phương tối thiểu có trọng số:
$$\min_{x_*, y_*} \sum_{u, i} c_{ui} (p_{ui} - x_u^T y_i)^2 + \lambda \left( \sum_u \|x_u\|^2 + \sum_i \|y_i\|^2 \right)$$
Trong đó:
*   $p_{ui}$ là chỉ số sở thích nhị phân: $p_{ui} = 1$ nếu tổng trọng số tương tác $r_{ui} > 0$, ngược lại $p_{ui} = 0$.
*   $c_{ui}$ là độ tin cậy (confidence measure): $c_{ui} = 1 + \alpha r_{ui}$ (với $\alpha$ là hằng số tỷ lệ, thường đặt mặc định là 40).
*   $\lambda$ là hệ số điều hòa (regularization) để tránh hiện tượng quá khớp (overfitting).
*   $x_u$ và $y_i$ lần lượt là vector đặc trưng ẩn đại diện cho người dùng $u$ và bộ phim $i$.

#### So sánh các giải thuật Collaborative Filtering:
| Giải pháp | Nguyên lý | Ưu điểm | Nhược điểm |
| :--- | :--- | :--- | :--- |
| **KNN Item-based** | Đo độ tương đồng giữa các cột của ma trận tương tác | Đơn giản, dễ giải thích. | Độ phức tạp tính toán rất lớn $O(I^2)$, không thể mở rộng (scale) khi số lượng phim tăng lên. |
| **Explicit SVD** | Phân rã ma trận dựa trên ratings tường minh (1-5 sao) | Hiệu quả cao khi dữ liệu rating dày đặc. | **Lỗi chệch dữ liệu**: Coi các phim chưa rate là nhãn âm (0), trong khi thực tế có thể user thích nhưng chưa biết phim đó. |
| **Implicit ALS** (Lựa chọn) | Phân rã ma trận dựa trên trọng số độ tin cậy ngầm định | Giải quyết triệt để bài toán thiếu nhãn âm, cực kỳ thích hợp cho log tương tác ngầm định (click/watch). | Cần tối ưu siêu tham số số chiều ẩn $f$ và hệ số $\alpha$. |

Tế bào này nhân ma trận tương tác thưa với trọng số confidence $\alpha=40$, huấn luyện mô hình ALS với 64 factors và lưu trữ kết quả.


In [3]:
# 2. Huấn luyện Implicit ALS model
alpha = 40
sparse_user_item = (user_item_matrix * alpha).astype('double')

model = implicit.als.AlternatingLeastSquares(
    factors=64,
    regularization=0.1,
    iterations=20,
    random_state=42
)

model.fit(sparse_user_item)

with open("models/als_model.pkl", "wb") as f:
    pickle.dump(model, f)
    
with open("processed_data/user_item_matrix.pkl", "wb") as f:
    pickle.dump(user_item_matrix, f)


C:\Users\Lenovo\anaconda3\Lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: Intel MKL BLAS is configured to use 14 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'MKL_NUM_THREADS=1' or by callng 'threadpoolctl.threadpool_limits(1, "blas")'. Having MKL use a threadpool can lead to severe performance issues
  check_blas_config()


  0%|          | 0/20 [00:00<?, ?it/s]

### Bước 4: Hàm lấy ứng viên Collaborative Filtering
Định nghĩa hàm `get_als_candidates` nhận vào ID người dùng, tìm chỉ mục mã hóa tương ứng, gọi hàm `recommend()` của mô hình ALS đã huấn luyện để dự đoán điểm số và lọc ra Top N phim chưa xem có điểm số cao nhất làm ứng viên.


In [4]:
# 3. Hàm đề xuất Collaborative Filtering candidates
def get_als_candidates(user_id, top_n=100):
    u_idx = user_to_idx.get(user_id, None)
    if u_idx is None:
        return movies_df.sort_values(by='popularity', ascending=False)['movieId'].head(top_n).tolist()
        
    ids, scores = model.recommend(u_idx, user_item_matrix[u_idx], N=top_n, filter_already_liked_items=True)
    recommended_movie_ids = [idx_to_movie[i] for i in ids]
    return recommended_movie_ids

# Thử nghiệm đề xuất
test_user = unique_users[0]
candidates = get_als_candidates(test_user, top_n=5)
print(f"Gợi ý ALS cho User {test_user}:", movies_df[movies_df['movieId'].isin(candidates)]['title'].tolist())


Gợi ý ALS cho User 1: ['Avatar: Fire and Ash', "Hippo's Revenge", 'Coco', 'Top Gun: Maverick', 'Final Destination Bloodlines']
